In [ ]:
deps_path = '/kaggle/input/datasets/nhhsag12/colpali-dependency'
!pip install --no-index --find-links {deps_path} --requirement {deps_path}/requirements.txt


In [ ]:
import os
import gc
import glob
import json
import pickle
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm

try:
    import pytrec_eval
except Exception:
    pytrec_eval = None

# ==============================================================================
# CONFIG — ColSmol model + ViDoRe data
# ==============================================================================

USE_PREENCODED_INDEX = True
PREENCODED_INDEX_SOURCE = "/kaggle/input/datasets/namthi/colsmol-encoded-vidore/vidore_colsmol_encoded_shards"
FALLBACK_INDEX_PKL_PATH = "/kaggle/input/datasets/namthi/colsmol-encoded-vidore/vidore_colsmol_encoded_shards/vidore_colsmol_manifest.pkl"
INDEX_PKL_PATH = PREENCODED_INDEX_SOURCE if USE_PREENCODED_INDEX else FALLBACK_INDEX_PKL_PATH

VIDORE_DATASET_ROOT = "/kaggle/input/datasets/namthi/vidore-v3"
DOMAIN_FILTER = []  # [] means all domains

# Keep ColSmol model setup unchanged
COLSMOL_BASE = "/kaggle/input/models/nhhsag12/colsmolvlm-instruct-500m-base/pytorch/default/1"
COLSMOL_LORA = "/kaggle/input/models/nhhsag12/colsmol-500m/pytorch/default/4"

WORKING_DIR = "/kaggle/working"
os.makedirs(WORKING_DIR, exist_ok=True)

# Method config
TOPK_RATIOS        = [round(i * 0.1, 1) for i in range(1, 10)]
N_LAST_LAYERS_LIST = [4, 8, 16, 32]
NORMALIZE_MODES    = ['pre', 'post']
ATTN_N_LAYERS_LIST = [1, 2, 4, 8]
KMEANS_ITERS       = 10
N_RANDOM_SEEDS     = 1

# Method 7 (RVQ) config
ENABLE_METHOD_7_RVQ = True
RVQ_CONFIGS = [
    (16, 256, "RVQ_NQ16_CB256"),
    (32, 256, "RVQ_NQ32_CB256"),
]
RVQ_TRAINING_SAMPLES    = 200_000
RVQ_TRAINING_EPOCHS     = 30
RVQ_TRAINING_BATCH_SIZE = 4096
RVQ_QUANTIZE_BATCH_SIZE = 8192
ADC_CHUNK_SIZE          = 1024
RERANK_TOP_K            = 100
M7_MEASURE_LATENCY      = True
RVQ_OFFLINE_WHEEL_DIR   = "/kaggle/input/datasets/thinam4/rvq-wheels/rvq_wheels"

# Strict benchmark mode (align with guide behavior)
USE_STRICT_GUIDE_EVAL = True
DEDUP_QUERIES_FOR_EVAL = True
EVAL_BACKEND = "pytrec_eval"   # options: pytrec_eval, fast_formula
AUTO_FALLBACK_WHEN_PYTREC_MISSING = True

print("Config loaded.")
print(f"INDEX_PKL_PATH      : {INDEX_PKL_PATH}")
print(f"VIDORE_DATASET_ROOT : {VIDORE_DATASET_ROOT}")
print(f"COLSMOL_BASE        : {COLSMOL_BASE}")
print(f"COLSMOL_LORA        : {COLSMOL_LORA}")
print(f"TOPK_RATIOS         : {TOPK_RATIOS}")
print(f"N_LAST_LAYERS_LIST  : {N_LAST_LAYERS_LIST}")
print(f"NORMALIZE_MODES     : {NORMALIZE_MODES}")
print(f"ATTN_N_LAYERS_LIST  : {ATTN_N_LAYERS_LIST}")
print(f"KMEANS_ITERS        : {KMEANS_ITERS}")
print(f"N_RANDOM_SEEDS      : {N_RANDOM_SEEDS}")
print(f"ENABLE_METHOD_7_RVQ : {ENABLE_METHOD_7_RVQ}")
print(f"RVQ_CONFIGS         : {RVQ_CONFIGS}")
print(f"USE_STRICT_GUIDE_EVAL: {USE_STRICT_GUIDE_EVAL}")
print(f"DEDUP_QUERIES_FOR_EVAL: {DEDUP_QUERIES_FOR_EVAL}")
print(f"EVAL_BACKEND         : {EVAL_BACKEND}")
if USE_STRICT_GUIDE_EVAL and EVAL_BACKEND == "pytrec_eval" and pytrec_eval is None:
    if AUTO_FALLBACK_WHEN_PYTREC_MISSING:
        print("[WARN] pytrec_eval not available. Falling back to fast_formula backend.")
        EVAL_BACKEND = "fast_formula"
    else:
        raise ImportError("pytrec_eval is required for strict guide eval. Install it first in this runtime.")
print(f"EVAL_BACKEND(final)  : {EVAL_BACKEND}")

In [ ]:
# Load encoded ViDoRe page embeddings (directory of shards, manifest, or single-file payload)
# ==============================================================================


def _resolve_existing_file(path_like, base_dir=None):
    s = str(path_like)
    candidates = []

    if os.path.isabs(s):
        candidates.append(s)
    elif base_dir:
        candidates.append(os.path.join(base_dir, s))

    if base_dir:
        candidates.append(os.path.join(base_dir, os.path.basename(s)))

    candidates.append(s)

    seen = set()
    for c in candidates:
        c_norm = os.path.normpath(c)
        if c_norm in seen:
            continue
        seen.add(c_norm)
        if os.path.isfile(c_norm):
            return c_norm
    return None


index_source = INDEX_PKL_PATH
raw_embeddings = []
page_keys = []
meta_records = []

if os.path.isdir(index_source):
    shard_files = sorted(glob.glob(os.path.join(index_source, "shard_*.pkl")))
    if not shard_files:
        raise ValueError(f"No shard_*.pkl files found under: {index_source}")

    print(f"Loading sharded index directly from directory: {index_source}")
    print(f"Detected shards: {len(shard_files)}")

    for sp in shard_files:
        with open(sp, "rb") as sf:
            shard = pickle.load(sf)
        raw_embeddings.extend(shard.get("fused_index", []))
        page_keys.extend(shard.get("page_keys", []))
        meta_records.extend(shard.get("metadata", []))

elif os.path.isfile(index_source):
    with open(index_source, "rb") as f:
        payload = pickle.load(f)

    if not isinstance(payload, dict):
        raise ValueError("Expected dict payload in encoded ViDoRe PKL.")

    if payload.get("format") == "vidore_sharded_v1":
        shard_files = payload.get("shard_files", [])
        if not shard_files:
            raise ValueError("Sharded payload has no shard_files.")

        manifest_dir = os.path.dirname(index_source)
        resolved_shards = []
        for sp in shard_files:
            resolved = _resolve_existing_file(sp, base_dir=manifest_dir)
            if resolved is None:
                raise FileNotFoundError(
                    f"Shard not found: {sp} (checked relative to {manifest_dir})"
                )
            resolved_shards.append(resolved)

        print(f"Loading sharded index from manifest: {index_source}")
        print(f"Resolved shards: {len(resolved_shards)}")

        for sp in resolved_shards:
            with open(sp, "rb") as sf:
                shard = pickle.load(sf)
            raw_embeddings.extend(shard.get("fused_index", []))
            page_keys.extend(shard.get("page_keys", []))
            meta_records.extend(shard.get("metadata", []))

    elif any(k in payload for k in ["fused_index", "embeddings"]):
        raw_embeddings = payload.get("fused_index", payload.get("embeddings", []))
        page_keys = payload.get("page_keys", [])
        meta_records = payload.get("metadata", [])

    else:
        raise ValueError("Unsupported index payload format.")

else:
    raise FileNotFoundError(f"Index source not found: {index_source}")

if not raw_embeddings:
    raise ValueError("No embeddings found in index payload.")

meta_df = pd.DataFrame(meta_records) if meta_records else pd.DataFrame()

all_page_embeddings = []
for emb in raw_embeddings:
    if isinstance(emb, torch.Tensor):
        arr = emb.detach().cpu().numpy()
    else:
        arr = np.asarray(emb)
    if arr.ndim != 2:
        raise ValueError(f"Embedding must be 2D (tokens x dim), got {arr.shape}")
    if arr.dtype not in (np.float16, np.float32):
        arr = arr.astype(np.float32, copy=False)
    all_page_embeddings.append(arr)

n_pages = len(all_page_embeddings)
all_page_indices = list(range(n_pages))

if len(meta_df) < n_pages:
    pad_rows = n_pages - len(meta_df)
    meta_df = pd.concat([meta_df, pd.DataFrame(index=range(pad_rows))], ignore_index=True)
meta_df = meta_df.iloc[:n_pages].copy()

if page_keys and len(page_keys) >= n_pages:
    meta_df["page_key"] = page_keys[:n_pages]
elif "page_key" not in meta_df.columns:
    meta_df["page_key"] = [f"page_{i}" for i in range(n_pages)]

if "join_doc_name" not in meta_df.columns:
    meta_df["join_doc_name"] = "unknown_doc"
if "safe_page" not in meta_df.columns:
    meta_df["safe_page"] = np.arange(n_pages, dtype=np.int32)
if "domain" not in meta_df.columns:
    meta_df["domain"] = "unknown"

meta_df["embed_idx"] = np.arange(n_pages, dtype=np.int32)
embedded_rows = meta_df.copy()
avail_docs = set(embedded_rows["join_doc_name"].astype(str).tolist())
doc_page_lookup = {doc: grp for doc, grp in embedded_rows.groupby("join_doc_name")}

print(f"Loaded encoded pages: {n_pages}")
print(f"Embedding shape sample: {all_page_embeddings[0].shape}")
print(f"Embedding dtype sample: {all_page_embeddings[0].dtype}")
print(f"Metadata columns: {list(embedded_rows.columns)}")
print(f"Docs in index: {len(avail_docs)}")

In [ ]:
from colpali_engine.models import ColIdefics3, ColIdefics3Processor
from peft import PeftModel

query_model = ColIdefics3.from_pretrained(
    COLSMOL_BASE,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    attn_implementation="eager" # or eager
)
query_model = PeftModel.from_pretrained(
    query_model,
    COLSMOL_LORA
).eval()
query_processor = ColIdefics3Processor.from_pretrained(COLSMOL_LORA)


In [ ]:
def forward_with_attentions_colpali(model, inputs):
    """
    Run ColPali forward pass and capture per-layer attention weights via hooks.

    Tìm transformer decoder layers bằng cách duyệt toàn bộ submodule thay vì
    hardcode path — tránh AttributeError khi cấu trúc PaliGemma thay đổi theo
    phiên bản transformers.

    Returns:
      proj        : (B, S, dim)  — từ ColPali.forward(), giống Method 1
      raw_outputs : SimpleNamespace với .attentions = tuple of (B, H, S, S) tensors
    """
    from types import SimpleNamespace

    captured_attns = []
    hooks = []

    # Tìm GemmaModel.layers bằng cách duyệt submodules thay vì hardcode path.
    # Với PaliGemmaForConditionalGeneration, cấu trúc thực tế là:
    #   model.model  (PaliGemmaForConditionalGeneration)
    #     .model     (PaliGemmaModel)
    #       .language_model (GemmaForCausalLM)
    #         .model (GemmaModel)
    #           .layers (ModuleList)
    # Nhưng có thể khác nhau tùy phiên bản — nên tìm động.
    layers = None
    candidates = [
        # path đúng nhất theo class ColPali trong notebook này
        lambda m: m.model.model.language_model.model.layers,
        # fallback 1: một số phiên bản bỏ wrapper .model bên trong
        lambda m: m.model.model.language_model.layers,
        # fallback 2: PaliGemmaForConditionalGeneration không có language_model attr
        #             mà dùng .language_model trực tiếp trên .model
        lambda m: m.model.language_model.model.layers,
        lambda m: m.model.language_model.layers,
    ]
    for try_fn in candidates:
        try:
            layers = try_fn(model)
            break
        except AttributeError:
            continue

    # Nếu vẫn không tìm thấy: quét toàn bộ named_modules để tìm ModuleList chứa self_attn
    if layers is None:
        for name, mod in model.named_modules():
            if (isinstance(mod, torch.nn.ModuleList)
                    and len(mod) > 0
                    and hasattr(mod[0], 'self_attn')):
                layers = mod
                break

    if layers is None:
        raise RuntimeError(
            "Không tìm được transformer decoder layers trong model ColPali. "
            "Kiểm tra lại cấu trúc model bằng: print(model)"
        )

    def _attn_hook(module, inp, out):
        # GemmaAttention trả về (attn_output, attn_weights, past_key_value)
        # attn_weights là (B, H, S, S) khi output_attentions=True, ngược lại là None
        if isinstance(out, tuple) and len(out) > 1 and out[1] is not None:
            captured_attns.append(out[1].detach())

    for layer in layers:
        hooks.append(layer.self_attn.register_forward_hook(_attn_hook))

    try:
        with torch.no_grad():
            kwargs = dict(inputs)
            kwargs['output_attentions'] = True
            proj = model(**kwargs)   # ColPali.forward → (B, S, dim)
    finally:
        for h in hooks:
            h.remove()

    raw_outputs = SimpleNamespace(
        attentions=tuple(captured_attns) if captured_attns else None
    )
    return proj, raw_outputs

# ==============================================================================
# Helper: encode_query_live
#
# Encodes a single query string and returns:
#   proj        : (1, S, dim)  L2-normalised, masked  [float32 on device]
#   raw_outputs : ModelOutput  (contains .attentions for Methods 2 & 4)
#   inputs      : BatchEncoding (contains attention_mask, input_ids)
#   encode_ms   : wall-clock encoding time in milliseconds
#
# process_queries() chain (from ColPaliProcessor source):
#   process_queries([q])
#     → prepends query_prefix ("") + appends pad_token × 10
#     → calls process_texts()
#     → tokenizer(bos_token + text, return_tensors="pt", padding="longest")
# ==============================================================================

def encode_query_live(question: str, processor, model, device: str):
    """
    Tokenise `question` with process_queries() and run a forward pass that
    also captures attention weights.  Used by Methods 2 and 4.
    Returns (proj, raw_outputs, inputs, encode_ms).
    """
    inputs = processor.process_queries([question]).to(device)

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.no_grad():
        proj, raw_outputs = forward_with_attentions_colpali(model, inputs)

    torch.cuda.synchronize()
    encode_ms = (time.perf_counter() - t0) * 1000.0

    return proj, raw_outputs, inputs, encode_ms

In [ ]:
# DEBUG CELL — run this before QA mapping cell (ViDoRe)
from pathlib import Path

root = Path(VIDORE_DATASET_ROOT)
if not root.exists():
    raise FileNotFoundError(f"ViDoRe dataset root not found: {VIDORE_DATASET_ROOT}")

domain_dirs = [p for p in sorted(root.iterdir()) if p.is_dir()]
if DOMAIN_FILTER:
    domain_dirs = [p for p in domain_dirs if p.name in set(DOMAIN_FILTER)]

print(f"ViDoRe root: {root}")
print(f"Domains selected ({len(domain_dirs)}): {[p.name for p in domain_dirs]}")
print(f"Indexed pages: {len(embedded_rows)}")

if "domain" in embedded_rows.columns:
    print("\nIndexed pages per domain:")
    print(embedded_rows.groupby("domain").size().sort_values(ascending=False).head(20))

query_parquets = []
for d in domain_dirs:
    for fp in d.rglob("*.parquet"):
        pstr = str(fp).replace("\\", "/").lower()
        if "/corpus/" in pstr:
            continue
        query_parquets.append(fp)

print(f"\nNon-corpus parquet files found: {len(query_parquets)}")
for fp in query_parquets[:20]:
    print(" -", fp)

if query_parquets:
    sample_df = pd.read_parquet(query_parquets[0])
    print("\nSample query parquet:", query_parquets[0])
    print("Columns:", list(sample_df.columns))
    print(sample_df.head(2).to_string())

In [ ]:
# ==============================================================================
# Build QA Pairs from ViDoRe (schema-driven, strict)
# ==============================================================================

from pathlib import Path
from collections import defaultdict

print("\nBuilding QA pairs from ViDoRe query files (schema-driven)...")

root = Path(VIDORE_DATASET_ROOT)
if not root.exists():
    raise FileNotFoundError(f"ViDoRe dataset root not found: {VIDORE_DATASET_ROOT}")

domain_dirs = [p for p in sorted(root.iterdir()) if p.is_dir()]
if DOMAIN_FILTER:
    domain_dirs = [p for p in domain_dirs if p.name in set(DOMAIN_FILTER)]

# Build strict corpus_id -> embed_idx mapping from encoded index metadata
if "corpus_id" not in embedded_rows.columns:
    raise ValueError(
        "`corpus_id` column is missing in encoded index metadata. "
        "ViDoRe qrels are keyed by corpus_id, so this mapping is required."
    )

if "domain" not in embedded_rows.columns:
    embedded_rows = embedded_rows.copy()
    embedded_rows["domain"] = "unknown"


def _norm_key(v):
    s = "" if pd.isna(v) else str(v).strip()
    if not s:
        return ""
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass
    return s


def _norm_domain(v):
    return "" if pd.isna(v) else str(v).strip().lower()


corpus_id_to_embed = defaultdict(list)
domain_corpus_id_to_embed = defaultdict(list)

for _, r in embedded_rows.iterrows():
    cid_key = _norm_key(r.get("corpus_id"))
    if not cid_key:
        continue

    d_key = _norm_domain(r.get("domain"))
    eidx = int(r["embed_idx"])

    corpus_id_to_embed[cid_key].append(eidx)
    domain_corpus_id_to_embed[(d_key, cid_key)].append(eidx)

if not corpus_id_to_embed:
    raise ValueError("No valid corpus_id found in encoded metadata.")

for k in list(corpus_id_to_embed.keys()):
    corpus_id_to_embed[k] = sorted(set(corpus_id_to_embed[k]))
for k in list(domain_corpus_id_to_embed.keys()):
    domain_corpus_id_to_embed[k] = sorted(set(domain_corpus_id_to_embed[k]))

qa_pairs = []

scan_files = 0
used_query_files = 0
used_qrels_files = 0

queries_total = 0
qrels_total = 0
qrels_positive = 0
qrels_mapped = 0

missing_qid_in_queries = 0
queries_without_positive_qrels = 0
queries_with_unmapped_corpus = 0

mapped_with_domain_key = 0
mapped_with_global_fallback = 0

for domain_dir in domain_dirs:
    domain_name = domain_dir.name
    domain_key = _norm_domain(domain_name)

    query_frames = []
    qrels_frames = []

    for fp in sorted(domain_dir.rglob("*.parquet")):
        pstr = str(fp).replace("\\", "/").lower()
        if "/corpus/" in pstr:
            continue

        scan_files += 1
        try:
            df = pd.read_parquet(fp)
        except Exception:
            continue

        if df is None or df.empty:
            continue

        if "/queries/" in pstr:
            if "query_id" in df.columns and "query" in df.columns:
                query_frames.append(df)
                used_query_files += 1
            continue

        if "/qrels/" in pstr:
            if "query_id" in df.columns and "corpus_id" in df.columns:
                qrels_frames.append(df)
                used_qrels_files += 1
            continue

    if not query_frames or not qrels_frames:
        continue

    queries_df = pd.concat(query_frames, ignore_index=True)
    qrels_df = pd.concat(qrels_frames, ignore_index=True)

    queries_total += len(queries_df)
    qrels_total += len(qrels_df)

    if "score" in qrels_df.columns:
        qrels_pos = qrels_df[pd.to_numeric(qrels_df["score"], errors="coerce") > 0].copy()
    else:
        qrels_pos = qrels_df.copy()
    qrels_positive += len(qrels_pos)

    # query_id -> list of (corpus_id, relevance_score)
    qid_to_cids = defaultdict(list)
    for _, r in qrels_pos.iterrows():
        qid_key = _norm_key(r.get("query_id"))
        cid_key = _norm_key(r.get("corpus_id"))
        rel_raw = r.get("score", 1)
        try:
            rel_score = float(rel_raw)
        except Exception:
            rel_score = 1.0
        if qid_key and cid_key:
            qid_to_cids[qid_key].append((cid_key, rel_score))

    # Build query_id -> GT embed indices (+ graded relevance per embed_idx)
    qid_to_gt = {}
    qid_to_gt_relevance = {}

    for qid_key, cid_score_pairs in qid_to_cids.items():
        gt = set()
        gt_relevance = {}
        mapped_any = False

        for cid_key, rel_score in cid_score_pairs:
            dom_pair = (domain_key, cid_key)
            if dom_pair in domain_corpus_id_to_embed:
                mapped_any = True
                mapped_indices = domain_corpus_id_to_embed[dom_pair]
                gt.update(mapped_indices)
                for eidx in mapped_indices:
                    gt_relevance[eidx] = max(float(gt_relevance.get(eidx, 0.0)), float(rel_score))
                mapped_with_domain_key += 1
                continue

            if cid_key in corpus_id_to_embed:
                mapped_any = True
                mapped_indices = corpus_id_to_embed[cid_key]
                gt.update(mapped_indices)
                for eidx in mapped_indices:
                    gt_relevance[eidx] = max(float(gt_relevance.get(eidx, 0.0)), float(rel_score))
                mapped_with_global_fallback += 1

        if mapped_any and gt:
            qid_to_gt[qid_key] = sorted(int(x) for x in gt)
            qid_to_gt_relevance[qid_key] = {int(k): float(v) for k, v in gt_relevance.items() if float(v) > 0}
            qrels_mapped += 1

    for _, r in queries_df.iterrows():
        qid_key = _norm_key(r.get("query_id"))
        if not qid_key:
            missing_qid_in_queries += 1
            continue

        qtext = r.get("query")
        question = "" if pd.isna(qtext) else str(qtext).strip()
        if not question:
            continue

        if qid_key not in qid_to_cids:
            queries_without_positive_qrels += 1
            continue

        gt_indices = qid_to_gt.get(qid_key, [])
        gt_relevance = qid_to_gt_relevance.get(qid_key, {})
        if not gt_indices:
            queries_with_unmapped_corpus += 1
            continue

        if "join_doc_name" in embedded_rows.columns and len(gt_indices) > 0:
            doc_name = str(embedded_rows.iloc[int(gt_indices[0])].get("join_doc_name", domain_name))
        else:
            doc_name = domain_name

        qa_pairs.append(
            {
                "question": question,
                "gt_embed_indices": gt_indices,
                "gt_relevance": gt_relevance,
                "doc_name": doc_name,
                "domain": domain_name,
            }
        )

qa_pairs_before_dedup = len(qa_pairs)
if DEDUP_QUERIES_FOR_EVAL:
    deduped = []
    seen_questions = set()
    for row in qa_pairs:
        q_key = str(row.get("question", "")).strip()
        if not q_key:
            continue
        if q_key in seen_questions:
            continue
        seen_questions.add(q_key)
        deduped.append(row)
    qa_pairs = deduped

print(f"Parquet files scanned            : {scan_files}")
print(f"Query files used                : {used_query_files}")
print(f"Qrels files used                : {used_qrels_files}")
print(f"Queries rows total              : {queries_total}")
print(f"Qrels rows total                : {qrels_total}")
print(f"Qrels rows positive             : {qrels_positive}")
print(f"Qrels query_ids mapped to index : {qrels_mapped}")
print(f"QA pairs built                  : {len(qa_pairs)}")
print(f"QA pairs deduplicated removed   : {qa_pairs_before_dedup - len(qa_pairs)}")
print(f"Queries w/o positive qrels      : {queries_without_positive_qrels}")
print(f"Queries unmapped corpus_id      : {queries_with_unmapped_corpus}")
print(f"Queries missing query_id        : {missing_qid_in_queries}")

print(f"Mapped by (domain, corpus_id)   : {mapped_with_domain_key}")
print(f"Mapped by global fallback       : {mapped_with_global_fallback}")

if not qa_pairs:
    raise ValueError(
        "No QA pairs built from strict query_id/corpus_id mapping. "
        "This indicates index metadata corpus_id is not aligned with qrels corpus_id."
    )

In [ ]:
# ==============================================================================
# Shared utilities: doc matrix builder, MaxSim, metrics, latency tracker
# ==============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


def build_doc_matrix(embeddings, device):
    """
    Convert list of np.ndarray embeddings to a padded (n_docs, max_len, D) tensor.
    Returns (doc_matrix, doc_mask).
    """
    arrays   = [torch.from_numpy(e).float() for e in embeddings]
    max_len  = max(a.shape[0] for a in arrays)
    D        = arrays[0].shape[1]
    n        = len(arrays)
    mat      = torch.zeros(n, max_len, D, dtype=torch.float32)
    mask     = torch.zeros(n, max_len, dtype=torch.bool)
    for i, a in enumerate(arrays):
        L = a.shape[0]
        mat[i, :L] = F.normalize(a, dim=-1)
        mask[i, :L] = True
    return mat.to(device), mask.to(device)


@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    """
    q_norm   : (N_q, D)   — query tokens, L2-normalized
    doc_matrix: (n_docs, max_len, D)
    doc_mask : (n_docs, max_len)  bool
    Returns  : (N_q, n_docs)  per-token MaxSim scores
    """
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)   # (N_q, n_docs, max_len)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values                             # (N_q, n_docs)


# ---------- Metrics ----------

def _normalize_gt_relevance(gt):
    # Accept dict[int,float] (graded qrels) or list/set of indices (binary relevance).
    if isinstance(gt, dict):
        out = {}
        for k, v in gt.items():
            try:
                kk = int(k)
                vv = float(v)
            except Exception:
                continue
            if vv > 0:
                out[kk] = vv
        return out

    out = {}
    for x in gt:
        try:
            xx = int(x)
            out[xx] = 1.0
        except Exception:
            continue
    return out

def recall_at_k(ranked, gt, k):
    gt_rel = _normalize_gt_relevance(gt)
    if not gt_rel:
        return 0.0
    denom = float(len(gt_rel))
    num = sum(1 for i in ranked[:k] if int(i) in gt_rel)
    return float(num) / denom if denom > 0 else 0.0

def compute_ndcg(ranked, gt, k):
    gt_rel = _normalize_gt_relevance(gt)
    if not gt_rel:
        return 0.0

    dcg = 0.0
    for r, i in enumerate(ranked[:k]):
        gain = float(gt_rel.get(int(i), 0.0))
        if gain > 0:
            dcg += gain / np.log2(r + 2)

    ideal_gains = sorted(gt_rel.values(), reverse=True)[:k]
    idcg = sum(float(g) / np.log2(r + 2) for r, g in enumerate(ideal_gains))
    return dcg / idcg if idcg > 0 else 0.0

def hit_metrics(top10, gt):
    gt_rel = _normalize_gt_relevance(gt)

    if USE_STRICT_GUIDE_EVAL and EVAL_BACKEND == "pytrec_eval":
        if pytrec_eval is None:
            raise ImportError("pytrec_eval backend requested but package is unavailable.")

        qrels = {"q0": {f"d{int(doc_id)}": float(rel) for doc_id, rel in gt_rel.items() if float(rel) > 0}}

        # Synthetic monotonic scores preserve rank order for pytrec_eval.
        results = {"q0": {f"d{int(doc_id)}": float(len(top10) - rank) for rank, doc_id in enumerate(top10)}}

        evaluator = pytrec_eval.RelevanceEvaluator(qrels, {"recall.1,5,10", "ndcg_cut.1,5,10"})
        s = evaluator.evaluate(results)["q0"]
        return {
            'r1':  float(s.get('recall_1', 0.0)),
            'r5':  float(s.get('recall_5', 0.0)),
            'r10': float(s.get('recall_10', 0.0)),
            'n1':  float(s.get('ndcg_cut_1', 0.0)),
            'n5':  float(s.get('ndcg_cut_5', 0.0)),
            'n10': float(s.get('ndcg_cut_10', 0.0)),
        }

    return {
        'r1':  float(recall_at_k(top10, gt_rel, 1)),
        'r5':  float(recall_at_k(top10, gt_rel, 5)),
        'r10': float(recall_at_k(top10, gt_rel, 10)),
        'n1':  float(compute_ndcg(top10, gt_rel, 1)),
        'n5':  float(compute_ndcg(top10, gt_rel, 5)),
        'n10': float(compute_ndcg(top10, gt_rel, 10)),
    }

def _init_metric():
    return {'r1': 0, 'r5': 0, 'r10': 0, 'n1': 0.0, 'n5': 0.0, 'n10': 0.0, 'count': 0}

def _add_metric(dst, src):
    dst['r1']    += float(src['r1'])
    dst['r5']    += float(src['r5'])
    dst['r10']   += float(src['r10'])
    dst['n1']    += float(src['n1'])
    dst['n5']    += float(src['n5'])
    dst['n10']   += float(src['n10'])
    dst['count'] += 1

def _ensure(store, key):
    if key not in store: store[key] = _init_metric()
    return store[key]

def record(all_metrics, all_domain_metrics, key, m, domain):
    _add_metric(_ensure(all_metrics, key), m)
    if domain not in all_domain_metrics:
        all_domain_metrics[domain] = {}
    _add_metric(_ensure(all_domain_metrics[domain], key), m)


def print_summary(all_metrics, all_domain_metrics, method_keys, title=""):
    if title:
        print(f"\n{'='*60}\n{title}\n{'='*60}")
    print(f"{'Method':<35} {'R@1':>7} {'R@5':>7} {'R@10':>7} {'nDCG@1':>8} {'nDCG@5':>8} {'nDCG@10':>9}")
    print("-" * 92)
    for key in method_keys:
        if key not in all_metrics: continue
        m = all_metrics[key]; cnt = m['count'] or 1
        print(f"{key:<35} {m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  "
              f"{m['r10']/cnt*100:6.2f}%  {m['n1']/cnt:7.4f}  {m['n5']/cnt:7.4f}  {m['n10']/cnt:8.4f}")


# ---------- Latency Tracker ----------

class LatencyTracker:
    """
    Tracks per-ratio latency of MaxSim scoring AFTER pooling/pruning.

    Only measures the time for: MaxSim scoring + aggregation + top-k
    on the already-reduced multi-vector representation.
    Excludes model forward pass, pooling, and pruning computation.

    Usage:
        tracker = LatencyTracker("Hierarchical Ward Pool")
        tracker.add_ratio(ratio, score_ms)   # inside loop, per ratio
        tracker.report()                     # after loop
    """
    def __init__(self, method_name: str):
        self.name = method_name
        self.ratio_ms = {}   # ratio -> list[float]  pool+retrieve ms per query

    def add_ratio(self, ratio: float, score_ms: float):
        """Record scoring latency for a single (ratio, query) observation."""
        if ratio not in self.ratio_ms:
            self.ratio_ms[ratio] = []
        self.ratio_ms[ratio].append(score_ms)

    def report(self):
        if not self.ratio_ms:
            print(f"[{self.name}] No latency data collected.")
            return
        print(f"\n{'='*70}")
        print(f"Latency Report — {self.name}  (scoring only, post-pooling/pruning)")
        print(f"{'='*70}")
        print(f"  {'Ratio':<10} {'n':>6} {'avg ms':>10} {'p50 ms':>10} {'p95 ms':>10}")
        print(f"  {'-'*50}")
        for ratio in sorted(self.ratio_ms.keys()):
            vals = self.ratio_ms[ratio]
            n    = len(vals)
            avg  = np.mean(vals)
            p50  = np.percentile(vals, 50)
            p95  = np.percentile(vals, 95)
            print(f"  {ratio:<10.0%} {n:>6} {avg:>10.2f} {p50:>10.2f} {p95:>10.2f}")

    def to_dict(self):
        rows = []
        for ratio in sorted(self.ratio_ms.keys()):
            vals = self.ratio_ms[ratio]
            n    = len(vals)
            rows.append({
                'method':           self.name,
                'ratio':            ratio,
                'n_queries':        n,
                'avg_score_ms':     round(np.mean(vals), 3) if n else 0,
                'p50_score_ms':     round(np.percentile(vals, 50), 3) if n else 0,
                'p95_score_ms':     round(np.percentile(vals, 95), 3) if n else 0,
            })
        return rows


# ==============================================================================
# Spherical KMeans (cosine similarity) — Method 5
# Input : X (N, D) L2-normalized query token vectors
# Output: centroids (K, D) — mean-pool each cluster then re-normalize
# ==============================================================================

def spherical_kmeans(X, K, n_iters=KMEANS_ITERS):
    """
    Cluster N query token vectors into K representative centroids
    using cosine distance (spherical KMeans).

    Args:
        X      : (N, D)  L2-normalized query token vectors
        K      : int     number of clusters (representative tokens to keep)
        n_iters: int     number of Lloyd-style iterations

    Returns:
        centroids: (K, D)  L2-normalized cluster centroids
    """
    N, D = X.shape
    K = min(K, N)

    if K >= N:
        return X.clone()   # every token is its own representative

    # Initialise: pick K distinct tokens at random
    perm      = torch.randperm(N, device=X.device)
    centroids = X[perm[:K]].clone()   # (K, D)

    for _ in range(n_iters):
        # Assignment: cosine sim = dot product when X is normalised
        sim         = torch.mm(X, centroids.t())   # (N, K)
        cluster_ids = sim.argmax(dim=1)             # (N,)

        # Update: mean-pool members → re-normalize
        new_centroids = torch.zeros_like(centroids)
        for c in range(K):
            members = (cluster_ids == c).nonzero(as_tuple=True)[0]
            new_centroids[c] = (X[members].mean(dim=0)
                                if members.numel() > 0 else centroids[c])
        centroids = F.normalize(new_centroids, dim=-1)

    return centroids   # (K, D)


# ==============================================================================
# Random Token Pruning — Method 6
# Randomly sample round(M * ratio) content tokens; average over N_RANDOM_SEEDS.
# ==============================================================================

def random_prune_topk(q_norm, content_idx, doc_matrix, doc_mask,
                      topk_ratios, n_seeds=N_RANDOM_SEEDS):
    """
    For each ratio r keep k = round(M * r) randomly selected tokens.
    Scores are averaged over `n_seeds` independent random draws to reduce
    variance.

    Timing covers the full scoring work per ratio:
      random sampling + MaxSim + sum  (averaged over n_seeds).
    This matches how all other methods measure latency.

    Args:
        q_norm      : (M, D)   content token vectors, L2-normalized
        content_idx : (M,)     positions of content tokens in the full sequence
                               (unused here; kept for API symmetry)
        doc_matrix  : (n_docs, max_len, D)
        doc_mask    : (n_docs, max_len)  bool
        topk_ratios : list[float]  fractions of tokens to KEEP
        n_seeds     : int  number of random samples to average

    Returns:
        results : dict {ratio: scores (n_docs,)}
        timing  : dict {ratio: score_ms}   ms for sampling + MaxSim per ratio
    """
    M      = q_norm.shape[0]
    n_docs = doc_matrix.shape[0]
    device = q_norm.device

    if M == 0:
        zero = torch.zeros(n_docs, device=device)
        return {r: zero for r in topk_ratios}, {r: 0.0 for r in topk_ratios}

    results = {}
    timing  = {}

    for ratio in topk_ratios:
        k = max(1, round(M * ratio))
        k = min(k, M)

        # ── Time: random sample + MaxSim + sum (all scoring work) ────────
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        if k == M:
            # Keep all tokens — no pruning needed; score as-is
            scores = fast_maxsim(q_norm, doc_matrix, doc_mask).sum(dim=0)
        else:
            accumulated = torch.zeros(n_docs, device=device, dtype=torch.float32)
            for _ in range(n_seeds):
                perm        = torch.randperm(M, device=device)[:k]
                pruned      = q_norm[perm]          # (k, D)
                accumulated += fast_maxsim(pruned, doc_matrix, doc_mask).sum(dim=0)
            scores = accumulated / n_seeds

        torch.cuda.synchronize()
        timing[ratio] = (time.perf_counter() - t0) * 1000.0
        # ─────────────────────────────────────────────────────────────────

        results[ratio] = scores

    return results, timing


print("Utility functions ready.")

In [ ]:
# ==============================================================================
# METHOD 1 — Traditional MaxSim
# Queries encoded live with ColPali (plain forward, no attention capture).
# ==============================================================================

print(">>> METHOD 1: Traditional MaxSim")

trad_metrics        = {}
trad_domain_metrics = {}
trad_query_rows     = []
trad_latency        = LatencyTracker("Traditional MaxSim")

METHOD_KEYS_TRAD = ['traditional']

# Build full doc matrix from all page embeddings
print(f"Building doc matrix from {len(all_page_embeddings)} page embeddings...")
doc_matrix, doc_mask = build_doc_matrix(all_page_embeddings, device)
n_docs = doc_matrix.shape[0]
print(f"Doc matrix shape: {doc_matrix.shape}")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Traditional MaxSim")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = item.get('gt_relevance', item['gt_embed_indices'])
    domain   = item['domain']

    # ── Encode query live ──────────────────────────────────────────────
    q_inputs = query_processor.process_queries([question]).to(device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # (1, S, dim)  — plain forward

    attn_mask = q_inputs['attention_mask'][0]                       # (S,)
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()                         # (N, dim)
    q_norm    = F.normalize(q_emb, dim=-1)

    # ── Retrieval (timed — this is the 100% baseline) ───────────────────
    torch.cuda.synchronize()
    t_score_start = time.perf_counter()

    M      = fast_maxsim(q_norm, doc_matrix, doc_mask)               # (N, n_docs)
    scores = M.sum(dim=0)
    top10  = torch.topk(scores, min(10, n_docs)).indices.cpu().tolist()

    torch.cuda.synchronize()
    score_ms = (time.perf_counter() - t_score_start) * 1000.0
    trad_latency.add_ratio(1.0, score_ms)

    m = hit_metrics(top10, gt_set)
    record(trad_metrics, trad_domain_metrics, 'traditional', m, domain)

    trad_query_rows.append({
        'query_id':       q_idx,
        'doc_name':       item['doc_name'],
        'domain':         domain,
        'question':       question,
        'trad_r@1':       m['r1'],
        'trad_r@5':       m['r5'],
        'trad_r@10':      m['r10'],
        'trad_ndcg@1':    round(m['n1'],  4),
        'trad_ndcg@5':    round(m['n5'],  4),
        'trad_ndcg@10':   round(m['n10'], 4),
    })

print_summary(trad_metrics, trad_domain_metrics, METHOD_KEYS_TRAD,
              title="Traditional MaxSim Results")
trad_latency.report()

pd.DataFrame(trad_query_rows).to_csv(
    os.path.join(WORKING_DIR, "traditional_queries.csv"), index=False)
print("\n✅ Saved: traditional_queries.csv")

In [ ]:
# ==============================================================================
# SVD / importance helpers
# ==============================================================================

SVD_RANK_REMOVE  = 1
TEMPERATURE_OURS = 0.5   # softplus temperature


def remove_sink_components_batch(attn_heads, k):
    try:
        U, S, Vh = torch.linalg.svd(attn_heads, full_matrices=False)
        k = min(k, S.shape[-1])
        sink = (U[..., :k] * S[:, :k].unsqueeze(1)) @ Vh[:, :k, :]
        return (attn_heads - sink).clamp(min=0.0)
    except Exception:
        return attn_heads


def compute_svd_importance_softplus(attentions, content_mask, layer_weights, k):
    """
    attentions  : list of (B, H, Sq, Sk) tensors — one per layer
    content_mask: (B, S) float tensor
    layer_weights: (n_layers,) tensor, exponentially increasing
    Returns     : (B, S) importance tensor
    """
    device_    = content_mask.device
    B, S       = content_mask.shape
    importance = torch.zeros(B, S, device=device_)

    for i, attn in enumerate(attentions):
        attn      = attn.float().to(device_)
        B_, H, Sq, Sk = attn.shape
        attn_flat = attn.view(B_ * H, Sq, Sk)
        cleaned   = remove_sink_components_batch(attn_flat, k)
        cleaned   = cleaned.view(B_, H, Sq, Sk)

        layer_imp = cleaned.sum(dim=2).mean(dim=1)          # (B, S)
        layer_imp = layer_imp * content_mask
        layer_imp = layer_imp / layer_imp.max(dim=-1, keepdim=True).values.clamp(min=1e-8)
        importance += layer_weights[i] * layer_imp

    importance = importance * content_mask
    importance = F.softplus(importance / TEMPERATURE_OURS)
    importance = importance * content_mask
    return importance


def build_content_mask_qwen(inputs, processor):
    """Mask out padding and special tokens from the attention mask."""
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()

    tok = getattr(processor, 'tokenizer', processor)
    special_ids = set()
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None:
            special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()

    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special     = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


def build_cluster_pool_vectors_ours(
    q_norm_kept, q_norm_disc,
    imp_kept, imp_disc,
    normalize_mode='pre',
):
    """
    Pooling step ONLY: assign discarded tokens to nearest kept token,
    then create weighted-average pooled vectors per cluster.

    Returns (pooled_vecs, pooled_weights) where:
      pooled_vecs   : (K, D) tensor — one pooled vector per cluster
      pooled_weights: (K,) tensor   — total importance weight per cluster
    If no discarded tokens, returns empty tensors.
    """
    P = q_norm_disc.shape[0]
    if P == 0:
        device_ = q_norm_kept.device
        D = q_norm_kept.shape[1]
        return torch.zeros(0, D, device=device_), torch.zeros(0, device=device_)

    sim_assign  = torch.mm(q_norm_disc, q_norm_kept.t())
    cluster_ids = sim_assign.argmax(dim=-1)

    pooled_vecs    = []
    pooled_weights = []
    for c in cluster_ids.unique():
        members  = (cluster_ids == c).nonzero(as_tuple=True)[0]
        w        = imp_disc[members]
        w_sum    = w.sum().clamp(min=1e-8)
        w_norm   = w / w_sum
        pool_vec = (q_norm_disc[members] * w_norm.unsqueeze(-1)).sum(dim=0)
        if normalize_mode == 'pre':
            pool_vec = F.normalize(pool_vec.unsqueeze(0), dim=-1).squeeze(0)
        pooled_vecs.append(pool_vec)
        pooled_weights.append(w_sum)

    return torch.stack(pooled_vecs), torch.stack(pooled_weights)


def make_ablation_keys_ours():
    keys = []
    for n in N_LAST_LAYERS_LIST:
        for mode in NORMALIZE_MODES:
            for r in TOPK_RATIOS:
                tag = f"L{n}_norm{mode}_r{int(r*100)}"
                keys.append(f"{tag}_trad")
                keys.append(f"{tag}_weighted")
    return keys

ABLATION_KEYS_OURS = make_ablation_keys_ours()

In [ ]:
# ==============================================================================
# Cell 7b — METHOD 2: Ours — SVD Importance + Cluster Pool (v12-Ablation)
# ==============================================================================

print(">>> METHOD 2: Ours — SVD Importance + Cluster Pool (v12-Ablation)")

ours_metrics        = {}
ours_domain_metrics = {}
ours_query_rows     = []
ours_latency        = LatencyTracker("Ours (SVD+ClusterPool)")

# doc_matrix / doc_mask / n_docs already built in Cell 6 — reuse them

print(f"Running ablation over {len(qa_pairs)} queries")
print(f"  n_last_layers : {N_LAST_LAYERS_LIST}")
print(f"  normalize_mode: {NORMALIZE_MODES}")
print(f"  topk_ratios   : {TOPK_RATIOS}")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Ours (SVD+ClusterPool)")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = item.get('gt_relevance', item['gt_embed_indices'])
    domain   = item['domain']

    # ── Encode query live (with attentions) ───────────────────────────
    proj, raw_outputs, q_inputs, encode_ms = encode_query_live(
        question, query_processor, query_model, device
    )

    attn_mask_1d    = q_inputs['attention_mask'][0].float()
    content_mask_1d = build_content_mask_qwen(q_inputs, query_processor)[0].float()

    trad_idx   = torch.where(attn_mask_1d   > 0)[0]
    method_idx = torch.where(content_mask_1d > 0)[0]
    if trad_idx.numel() == 0:
        continue
    if method_idx.numel() == 0:
        method_idx = trad_idx

    content_mask_2d = content_mask_1d.unsqueeze(0)
    embed_cache = {}   # n_layers -> (proj_1d, imp_1d)

    for n_layers in N_LAST_LAYERS_LIST:
        if raw_outputs.attentions is not None and len(raw_outputs.attentions) > 0:
            attn_list = list(raw_outputs.attentions[-n_layers:])
            n_actual  = len(attn_list)
        else:
            attn_list = []
            n_actual  = 0

        layer_weights = torch.exp(
            torch.linspace(0, 1, max(n_actual, 1), device=device)
        )
        layer_weights /= layer_weights.sum()

        if n_actual > 0:
            importance = compute_svd_importance_softplus(
                attn_list, content_mask_2d, layer_weights, SVD_RANK_REMOVE
            )
        else:
            importance = content_mask_2d.clone()

        importance = (importance * content_mask_2d)[0]
        embed_cache[n_layers] = (proj[0].float(), importance)

    query_row = {
        'query_id':      q_idx,
        'doc_name':      item['doc_name'],
        'domain':        domain,
        'question':      question,
    }

    # Ablation: n_last_layers × normalize_mode × ratio × scoring_mode
    for n_layers in N_LAST_LAYERS_LIST:
        q_embed, q_importance = embed_cache[n_layers]

        q_method_norm  = F.normalize(q_embed[method_idx].float(), dim=-1)

        imp_valid      = q_importance[method_idx].float()
        n_method       = method_idx.numel()
        sorted_imp_idx = torch.argsort(imp_valid, descending=True)

        for mode in NORMALIZE_MODES:
            for r in TOPK_RATIOS:
                tag    = f"L{n_layers}_norm{mode}_r{int(r*100)}"
                n_keep = max(1, int(n_method * r))

                # ── Pooling/pruning (NOT timed) ────────────────────
                kept_idx = sorted_imp_idx[:n_keep]
                disc_idx = sorted_imp_idx[n_keep:]

                q_kept   = q_method_norm[kept_idx]
                q_disc   = q_method_norm[disc_idx]
                imp_kept = imp_valid[kept_idx]
                imp_disc = imp_valid[disc_idx]

                pooled_vecs, pooled_weights = build_cluster_pool_vectors_ours(
                    q_kept, q_disc, imp_kept, imp_disc,
                    normalize_mode=mode,
                )
                total_disc_imp = imp_disc.sum().item()
                pool_frac = total_disc_imp / imp_valid.sum().clamp(min=1e-8).item()

                # Combine kept + pooled into a single set of query vectors
                if pooled_vecs.shape[0] > 0:
                    all_q_vecs = torch.cat([q_kept, pooled_vecs], dim=0)
                    all_q_weights = torch.cat([
                        imp_kept,
                        pooled_weights * pool_frac,
                    ])
                    n_kept_vecs = q_kept.shape[0]
                else:
                    all_q_vecs = q_kept
                    all_q_weights = imp_kept
                    n_kept_vecs = q_kept.shape[0]

                # ── Time MaxSim scoring on kept + pooled vectors ───
                torch.cuda.synchronize()
                t_score_start = time.perf_counter()

                M_all    = fast_maxsim(all_q_vecs, doc_matrix, doc_mask)

                # trad scoring: sum of kept MaxSim + sum of pooled MaxSim * pool_frac
                M_kept_part   = M_all[:n_kept_vecs]
                M_pooled_part = M_all[n_kept_vecs:]
                sc_trad = M_kept_part.sum(dim=0)
                if M_pooled_part.shape[0] > 0:
                    sc_trad = sc_trad + (M_pooled_part * pooled_weights.unsqueeze(-1)).sum(dim=0) / max(total_disc_imp, 1e-8) * pool_frac
                top10_trad = torch.topk(sc_trad, min(10, n_docs)).indices.cpu().tolist()

                # weighted scoring
                all_w_n  = all_q_weights / all_q_weights.sum().clamp(min=1e-8)
                sc_wt    = (M_all * all_w_n.unsqueeze(-1)).sum(dim=0)
                top10_wt = torch.topk(sc_wt, min(10, n_docs)).indices.cpu().tolist()

                torch.cuda.synchronize()
                score_ms = (time.perf_counter() - t_score_start) * 1000.0
                ours_latency.add_ratio(r, score_ms)
                # ────────────────────────────────────────────────────

                m_trad     = hit_metrics(top10_trad, gt_set)
                key_trad   = f"{tag}_trad"
                record(ours_metrics, ours_domain_metrics, key_trad, m_trad, domain)
                query_row.update({
                    f'{key_trad}_r@1':    m_trad['r1'],
                    f'{key_trad}_r@5':    m_trad['r5'],
                    f'{key_trad}_r@10':   m_trad['r10'],
                    f'{key_trad}_ndcg@10':round(m_trad['n10'], 4),
                })

                m_wt       = hit_metrics(top10_wt, gt_set)
                key_wt     = f"{tag}_weighted"
                record(ours_metrics, ours_domain_metrics, key_wt, m_wt, domain)
                query_row.update({
                    f'{key_wt}_r@1':    m_wt['r1'],
                    f'{key_wt}_r@5':    m_wt['r5'],
                    f'{key_wt}_r@10':   m_wt['r10'],
                    f'{key_wt}_ndcg@10':round(m_wt['n10'], 4),
                })

    ours_query_rows.append(query_row)

    if q_idx % 50 == 0:
        gc.collect()
        torch.cuda.empty_cache()

print_summary(ours_metrics, ours_domain_metrics,
              ABLATION_KEYS_OURS,
              title="Ours — SVD Importance + Cluster Pool")
ours_latency.report()

pd.DataFrame(ours_query_rows).to_csv(
    os.path.join(WORKING_DIR, "ours_ablation_queries.csv"), index=False)
print("\n✅ Saved: ours_ablation_queries.csv")

In [ ]:
# ==============================================================================
# METHOD 3 — Hierarchical: Ward agglomerative token pooling
# ==============================================================================

print(">>> METHOD 3: Hierarchical Ward Token Pooling")


def _ward_distance_matrix(cents, sizes):
    device_  = cents.device
    sim      = torch.matmul(cents, cents.t()).clamp(-1.0, 1.0)
    sq_dist  = 2.0 * (1.0 - sim)
    ni = sizes.float().unsqueeze(1)
    nj = sizes.float().unsqueeze(0)
    w  = (ni * nj) / (ni + nj)
    ward = w * sq_dist
    mask = torch.ones(cents.shape[0], cents.shape[0], dtype=torch.bool, device=device_).tril()
    ward.masked_fill_(mask, float('inf'))
    return ward


def ward_pool(vecs, n_clusters):
    """vecs: (N, D) L2-normalized. Returns centroids (K, D) L2-normalized."""
    N = vecs.shape[0]
    if N <= n_clusters:
        return vecs.clone()

    device_ = vecs.device
    sums    = vecs.clone().float()
    sizes   = torch.ones(N, device=device_, dtype=torch.float32)
    cents   = F.normalize(sums, dim=-1)
    active  = torch.ones(N, dtype=torch.bool, device=device_)
    C       = N

    while C > n_clusters:
        active_idx = active.nonzero(as_tuple=True)[0]
        c_act      = cents[active_idx]
        s_act      = sizes[active_idx]
        ward       = _ward_distance_matrix(c_act, s_act)
        flat_idx   = ward.argmin().item()
        n_act      = active_idx.shape[0]
        ai, aj     = flat_idx // n_act, flat_idx % n_act
        gi, gj     = active_idx[ai].item(), active_idx[aj].item()
        sums[gi]   = sums[gi] + sums[gj]
        sizes[gi]  = sizes[gi] + sizes[gj]
        cents[gi]  = F.normalize(sums[gi].unsqueeze(0), dim=-1).squeeze(0)
        active[gj] = False
        C -= 1

    final_idx = active.nonzero(as_tuple=True)[0]
    return cents[final_idx]


def ward_pool_scores_all_ratios(q_norm, doc_matrix, doc_mask, topk_ratios):
    """
    Returns (results, timing) where:
      results : dict[ratio -> scores tensor]
      timing  : dict[ratio -> score_ms]  (MaxSim scoring only, excludes ward pooling)
    """
    N       = q_norm.shape[0]
    results = {}
    timing  = {}
    prev_vecs, prev_C = q_norm.clone(), N

    for ratio in sorted(topk_ratios, reverse=True):
        # ── Ward pooling (NOT timed) ──────────────────────────
        target_C = max(1, round(N * ratio))
        if target_C >= prev_C:
            centroids = prev_vecs
        else:
            centroids = ward_pool(prev_vecs, target_C)
            prev_vecs, prev_C = centroids, centroids.shape[0]

        # ── Time ONLY MaxSim scoring after pooling ────────────
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        M_c            = fast_maxsim(centroids, doc_matrix, doc_mask)
        results[ratio] = M_c.sum(0)

        torch.cuda.synchronize()
        timing[ratio] = (time.perf_counter() - t0) * 1000.0

    return results, timing


# ---------- Keys ----------
METHOD_KEYS_HIER = ['traditional'] + [f"hier_r{int(r*100)}" for r in TOPK_RATIOS]

hier_metrics        = {}
hier_domain_metrics = {}
hier_query_rows     = []
hier_latency        = LatencyTracker("Hierarchical Ward Pool")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Hierarchical Ward Pool")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = item.get('gt_relevance', item['gt_embed_indices'])
    domain   = item['domain']

    # ── Encode query live ──────────────────────────────────────────────
    q_inputs = query_processor.process_queries([question]).to(device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # plain forward

    attn_mask = q_inputs['attention_mask'][0]
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()
    q_norm    = F.normalize(q_emb, dim=-1)

    # ── Retrieval ──────────────────────────────────────────────────────

    # Baseline traditional
    M_trad     = fast_maxsim(q_norm, doc_matrix, doc_mask)
    trad_sc    = M_trad.sum(dim=0)
    top10_trad = torch.topk(trad_sc, min(10, n_docs)).indices.cpu().tolist()
    m_trad     = hit_metrics(top10_trad, gt_set)
    record(hier_metrics, hier_domain_metrics, 'traditional', m_trad, domain)

    # Ward hierarchical pooling — all ratios, with per-ratio timing
    ratio_scores, ratio_timing = ward_pool_scores_all_ratios(q_norm, doc_matrix, doc_mask, TOPK_RATIOS)

    query_row = {
        'query_id': q_idx, 'doc_name': item['doc_name'],
        'domain': domain,  'question': question,
        'trad_r@1': m_trad['r1'],   'trad_r@5': m_trad['r5'],
        'trad_r@10': m_trad['r10'], 'trad_ndcg@10': round(m_trad['n10'], 4),
    }

    for r in TOPK_RATIOS:
        key   = f"hier_r{int(r*100)}"
        top10 = torch.topk(ratio_scores[r], min(10, n_docs)).indices.cpu().tolist()
        m     = hit_metrics(top10, gt_set)
        record(hier_metrics, hier_domain_metrics, key, m, domain)
        hier_latency.add_ratio(r, ratio_timing[r])
        query_row.update({
            f'{key}_r@1':     m['r1'],
            f'{key}_r@5':     m['r5'],
            f'{key}_r@10':    m['r10'],
            f'{key}_ndcg@10': round(m['n10'], 4),
        })

    hier_query_rows.append(query_row)

print_summary(hier_metrics, hier_domain_metrics, METHOD_KEYS_HIER,
              title="Hierarchical Ward Pooling Results")
hier_latency.report()

pd.DataFrame(hier_query_rows).to_csv(
    os.path.join(WORKING_DIR, "hierarchical_queries.csv"), index=False)
print("\n✅ Saved: hierarchical_queries.csv")

In [ ]:
# ==============================================================================
# METHOD 4 — Attention Score Token Pruning (v13)
#
# Per-token importance = column-sum of attention weights across all heads/layers.
# Queries are encoded live with forward_with_attentions; latency is measured.
# ==============================================================================

print(">>> METHOD 4: Attention Score Token Pruning (Lassance et al. 2021)")

# ---------- Keys ----------
METHOD_KEYS_ATTN = ['traditional']
for n in ATTN_N_LAYERS_LIST:
    for r in TOPK_RATIOS:
        METHOD_KEYS_ATTN += [
            f"attn_L{n}_r{int(r*100)}_trad",
            f"attn_L{n}_r{int(r*100)}_weighted",
        ]

attn_metrics        = {}
attn_domain_metrics = {}
attn_query_rows     = []
attn_latency        = LatencyTracker("Attention Score Pruning")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Attention Score Pruning")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = item.get('gt_relevance', item['gt_embed_indices'])
    domain   = item['domain']

    # ── Encode query live (with attentions) ───────────────────────────
    proj, raw_outputs, q_inputs, encode_ms = encode_query_live(
        question, query_processor, query_model, device
    )

    attn_mask_1d = q_inputs['attention_mask'][0].float()
    trad_idx     = torch.where(attn_mask_1d > 0)[0]
    N            = trad_idx.numel()

    q_emb  = proj[0][trad_idx].float()
    q_norm = F.normalize(q_emb, dim=-1)

    # Baseline (full tokens)
    M_full     = fast_maxsim(q_norm, doc_matrix, doc_mask)
    trad_sc    = M_full.sum(dim=0)
    top10_trad = torch.topk(trad_sc, min(10, n_docs)).indices.cpu().tolist()
    m_trad     = hit_metrics(top10_trad, gt_set)
    record(attn_metrics, attn_domain_metrics, 'traditional', m_trad, domain)

    query_row = {
        'query_id': q_idx, 'doc_name': item['doc_name'],
        'domain': domain,  'question': question,
        'trad_r@1': m_trad['r1'],   'trad_r@5': m_trad['r5'],
        'trad_r@10': m_trad['r10'], 'trad_ndcg@10': round(m_trad['n10'], 4),
    }

    for n_layers in ATTN_N_LAYERS_LIST:
        # Compute attention column-sum importance from the last n_layers attention tensors
        # i(t) = Σ_h Σ_j A_{h,j,t}  (sum of attention received by token t, Lassance 2021)
        if raw_outputs.attentions is not None and len(raw_outputs.attentions) >= 1:
            attn_subset = list(raw_outputs.attentions[-n_layers:])
            # Each element: (1, H, Sq, Sk)  → sum over query dim → (1, H, Sk) → mean heads → (1, Sk)
            imp_2d = torch.zeros(1, q_inputs['attention_mask'].shape[1], device=device)
            for attn_layer in attn_subset:
                col_sum = attn_layer.float().sum(dim=2).mean(dim=1)  # (1, S)
                imp_2d += col_sum
            imp_full = imp_2d[0]                                      # (S,)
            imp      = imp_full[trad_idx]                             # (N,)
        else:
            imp = torch.ones(N, device=device)

        sorted_imp_idx = torch.argsort(imp, descending=True)

        for r in TOPK_RATIOS:
            n_keep   = max(1, int(N * r))

            # ── Pruning (NOT timed) ────────────────────────────
            kept_idx = sorted_imp_idx[:n_keep]
            imp_kept = imp[kept_idx]
            q_kept   = q_norm[kept_idx]                          # pruned vectors

            # ── Time MaxSim scoring on pruned vectors ──────────
            torch.cuda.synchronize()
            t_score_start = time.perf_counter()

            M_kept   = fast_maxsim(q_kept, doc_matrix, doc_mask) # (n_keep, n_docs)
            sc_trad  = M_kept.sum(dim=0)
            top_t    = torch.topk(sc_trad, min(10, n_docs)).indices.cpu().tolist()

            imp_k_n  = imp_kept / imp_kept.sum().clamp(min=1e-8)
            sc_w     = (M_kept * imp_k_n.unsqueeze(-1)).sum(dim=0)
            top_w    = torch.topk(sc_w, min(10, n_docs)).indices.cpu().tolist()

            torch.cuda.synchronize()
            score_ms = (time.perf_counter() - t_score_start) * 1000.0
            attn_latency.add_ratio(r, score_ms)
            # ────────────────────────────────────────────────────

            m_t      = hit_metrics(top_t, gt_set)
            key_t    = f"attn_L{n_layers}_r{int(r*100)}_trad"
            record(attn_metrics, attn_domain_metrics, key_t, m_t, domain)
            query_row.update({f'{key_t}_r@1': m_t['r1'], f'{key_t}_r@5': m_t['r5'],
                              f'{key_t}_r@10': m_t['r10'], f'{key_t}_ndcg@10': round(m_t['n10'], 4)})

            m_w      = hit_metrics(top_w, gt_set)
            key_w    = f"attn_L{n_layers}_r{int(r*100)}_weighted"
            record(attn_metrics, attn_domain_metrics, key_w, m_w, domain)
            query_row.update({f'{key_w}_r@1': m_w['r1'], f'{key_w}_r@5': m_w['r5'],
                              f'{key_w}_r@10': m_w['r10'], f'{key_w}_ndcg@10': round(m_w['n10'], 4)})

    attn_query_rows.append(query_row)

print_summary(attn_metrics, attn_domain_metrics,
              ['traditional'] + [f"attn_L1_r{int(r*100)}_trad" for r in TOPK_RATIOS],
              title="Attention Score Pruning (L=1, trad scoring)")
attn_latency.report()

pd.DataFrame(attn_query_rows).to_csv(
    os.path.join(WORKING_DIR, "attention_queries.csv"), index=False)
print("\n✅ Saved: attention_queries.csv")

In [ ]:
# ==============================================================================
# METHOD 5 — Spherical KMeans Token Pooling
#
# Groups the N query tokens into K = max(1, int(N * ratio)) clusters using
# spherical KMeans (cosine distance).  Each cluster is represented by its
# L2-normalised mean-pooled centroid.  MaxSim is then run with K centroid
# vectors instead of N raw token vectors.
#
# References: PLAID / ColBERT v2 cluster-pruning idea adapted for query-side.
# Queries are encoded live with the plain forward pass (no attentions needed).
# ==============================================================================

print(">>> METHOD 5: Spherical KMeans Token Pooling")

# ---------- Keys ----------
METHOD_KEYS_KMEANS = ['traditional'] + [f"kmeans_r{int(r*100)}" for r in TOPK_RATIOS]

kmeans_metrics        = {}
kmeans_domain_metrics = {}
kmeans_query_rows     = []
kmeans_latency        = LatencyTracker("Spherical KMeans Pool")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Spherical KMeans Pool")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = item.get('gt_relevance', item['gt_embed_indices'])
    domain   = item['domain']

    # ── Encode query live (plain forward — no attentions needed) ──────────
    q_inputs = query_processor.process_queries([question]).to(device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # (1, S, dim)

    attn_mask = q_inputs['attention_mask'][0]
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()
    q_norm    = F.normalize(q_emb, dim=-1)          # (N, D)
    N         = q_norm.shape[0]

    # ── Baseline (full tokens) ────────────────────────────────────────────
    M_full     = fast_maxsim(q_norm, doc_matrix, doc_mask)
    trad_sc    = M_full.sum(dim=0)
    top10_trad = torch.topk(trad_sc, min(10, n_docs)).indices.cpu().tolist()
    m_trad     = hit_metrics(top10_trad, gt_set)
    record(kmeans_metrics, kmeans_domain_metrics, 'traditional', m_trad, domain)

    query_row = {
        'query_id': q_idx, 'doc_name': item['doc_name'],
        'domain': domain,  'question': question,
        'trad_r@1': m_trad['r1'],   'trad_r@5': m_trad['r5'],
        'trad_r@10': m_trad['r10'], 'trad_ndcg@10': round(m_trad['n10'], 4),
    }

    # ── Spherical KMeans pooling — all ratios ─────────────────────────────
    for r in TOPK_RATIOS:
        K = max(1, int(N * r))

        # ── KMeans pooling (NOT timed) ────────────────────────────
        centroids = spherical_kmeans(q_norm, K)       # (K, D)

        # ── Time MaxSim scoring on centroids ──────────────────────
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        M_k   = fast_maxsim(centroids, doc_matrix, doc_mask)  # (K, n_docs)
        scores = M_k.sum(dim=0)
        top10  = torch.topk(scores, min(10, n_docs)).indices.cpu().tolist()

        torch.cuda.synchronize()
        score_ms = (time.perf_counter() - t0) * 1000.0
        kmeans_latency.add_ratio(r, score_ms)
        # ────────────────────────────────────────────────────────────

        key = f"kmeans_r{int(r*100)}"
        m   = hit_metrics(top10, gt_set)
        record(kmeans_metrics, kmeans_domain_metrics, key, m, domain)
        query_row.update({
            f'{key}_r@1':     m['r1'],
            f'{key}_r@5':     m['r5'],
            f'{key}_r@10':    m['r10'],
            f'{key}_ndcg@10': round(m['n10'], 4),
        })

    kmeans_query_rows.append(query_row)

print_summary(kmeans_metrics, kmeans_domain_metrics, METHOD_KEYS_KMEANS,
              title="Spherical KMeans Pooling Results")
kmeans_latency.report()

pd.DataFrame(kmeans_query_rows).to_csv(
    os.path.join(WORKING_DIR, "kmeans_queries.csv"), index=False)
print("\n✅ Saved: kmeans_queries.csv")

In [ ]:
# ==============================================================================
# METHOD 6 — Random Token Pruning (v14)
#
# Ablation baseline: instead of using any importance signal, tokens are sampled
# UNIFORMLY AT RANDOM.  If random pruning performs comparably to attention-based
# or SVD-based pruning, the importance signal offers little value.
#
# For each (query, ratio) pair: sample round(N * ratio) content tokens
# N_RANDOM_SEEDS times and average the resulting MaxSim scores to reduce
# variance from the random draw.
#
# Queries are encoded live with the plain forward pass (no attentions needed).
# ==============================================================================

print(">>> METHOD 6: Random Token Pruning (Ablation Baseline)")

# ---------- Keys ----------
METHOD_KEYS_RAND = ['traditional'] + [f"rand_r{int(r*100)}" for r in TOPK_RATIOS]

rand_metrics        = {}
rand_domain_metrics = {}
rand_query_rows     = []
rand_latency        = LatencyTracker("Random Pruning")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Random Pruning")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = item.get('gt_relevance', item['gt_embed_indices'])
    domain   = item['domain']

    # ── Encode query live (plain forward — no attentions needed) ──────────
    q_inputs = query_processor.process_queries([question]).to(device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # (1, S, dim)

    attn_mask = q_inputs['attention_mask'][0]
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()
    q_norm    = F.normalize(q_emb, dim=-1)   # (N, D) — content tokens only

    # ── Baseline (full tokens) ────────────────────────────────────────────
    M_full     = fast_maxsim(q_norm, doc_matrix, doc_mask)
    trad_sc    = M_full.sum(dim=0)
    top10_trad = torch.topk(trad_sc, min(10, n_docs)).indices.cpu().tolist()
    m_trad     = hit_metrics(top10_trad, gt_set)
    record(rand_metrics, rand_domain_metrics, 'traditional', m_trad, domain)

    query_row = {
        'query_id': q_idx, 'doc_name': item['doc_name'],
        'domain': domain,  'question': question,
        'trad_r@1': m_trad['r1'],   'trad_r@5': m_trad['r5'],
        'trad_r@10': m_trad['r10'], 'trad_ndcg@10': round(m_trad['n10'], 4),
        'n_random_seeds': N_RANDOM_SEEDS,
    }

    # ── Random pruning — all ratios, averaged over N_RANDOM_SEEDS ─────────
    # Timing is measured inside random_prune_topk per ratio, covering:
    # random sampling + MaxSim + sum  (the actual scoring work).
    ratio_scores, ratio_timing = random_prune_topk(
        q_norm      = q_norm,
        content_idx = trad_idx,          # passed for API symmetry; not used internally
        doc_matrix  = doc_matrix,
        doc_mask    = doc_mask,
        topk_ratios = TOPK_RATIOS,
        n_seeds     = N_RANDOM_SEEDS,
    )

    for r in TOPK_RATIOS:
        key    = f"rand_r{int(r*100)}"
        scores = ratio_scores[r]
        rand_latency.add_ratio(r, ratio_timing[r])

        top10 = torch.topk(scores, min(10, n_docs)).indices.cpu().tolist()
        m = hit_metrics(top10, gt_set)
        record(rand_metrics, rand_domain_metrics, key, m, domain)
        query_row.update({
            f'{key}_r@1':     m['r1'],
            f'{key}_r@5':     m['r5'],
            f'{key}_r@10':    m['r10'],
            f'{key}_ndcg@10': round(m['n10'], 4),
        })

    rand_query_rows.append(query_row)

print_summary(rand_metrics, rand_domain_metrics, METHOD_KEYS_RAND,
              title="Random Token Pruning Results")
rand_latency.report()

pd.DataFrame(rand_query_rows).to_csv(
    os.path.join(WORKING_DIR, "random_pruning_queries.csv"), index=False)
print("\n✅ Saved: random_pruning_queries.csv")

In [ ]:
# ==============================================================================
# METHOD 7 — Residual Vector Quantization (RVQ) Compression
# ==============================================================================

if ENABLE_METHOD_7_RVQ:
    print(">>> METHOD 7: Residual Vector Quantization (RVQ) Compression")

    import subprocess
    if os.path.isdir(RVQ_OFFLINE_WHEEL_DIR):
        subprocess.run([
            "pip", "install", "--quiet",
            "--no-index",
            "--find-links", RVQ_OFFLINE_WHEEL_DIR,
            "vector-quantize-pytorch",
        ], check=True)
        print(f"✅ Installed vector-quantize-pytorch from {RVQ_OFFLINE_WHEEL_DIR}")
    else:
        print(f"⚠️  Wheel dir not found: {RVQ_OFFLINE_WHEEL_DIR}, assuming already installed")

    from vector_quantize_pytorch import ResidualVQ

    device = "cuda" if torch.cuda.is_available() else "cpu"
    n_pages_total = len(all_page_embeddings)
    EMB_DIM = all_page_embeddings[0].shape[-1]
    print(f"Pages: {n_pages_total}, Dim: {EMB_DIM}")

    def _sample_patch_embeddings(embeddings_list, n_samples, seed=42):
        rng = np.random.default_rng(seed)
        all_lengths = [e.shape[0] for e in embeddings_list]
        total_patches = sum(all_lengths)
        n_take = min(n_samples, total_patches)

        flat_idx = rng.choice(total_patches, size=n_take, replace=False)
        flat_idx.sort()

        result = np.empty((n_take, embeddings_list[0].shape[1]), dtype=np.float32)
        cumsum = 0
        out_pos = 0
        flat_iter = iter(flat_idx)
        next_fi = next(flat_iter, None)

        for page_i, L in enumerate(all_lengths):
            page_end = cumsum + L
            while next_fi is not None and next_fi < page_end:
                local = next_fi - cumsum
                result[out_pos] = embeddings_list[page_i][local].astype(np.float32)
                out_pos += 1
                next_fi = next(flat_iter, None)
            cumsum = page_end
            if next_fi is None:
                break

        return torch.from_numpy(result[:out_pos])

    def _extract_codebooks(rvq_model, NQ, device_):
        codebooks = []
        for q_idx in range(NQ):
            vq_layer = rvq_model.layers[q_idx]
            cb = None

            if cb is None:
                try:
                    embed = vq_layer._codebook.embed
                    if embed.ndim == 3:
                        cb = embed[0]
                    elif embed.ndim == 2:
                        cb = embed
                except (AttributeError, IndexError):
                    pass

            if cb is None:
                try:
                    embed = vq_layer.codebook
                    if embed.ndim == 3:
                        cb = embed[0]
                    elif embed.ndim == 2:
                        cb = embed
                except AttributeError:
                    pass

            if cb is None:
                try:
                    embed = vq_layer._codebook.embed_avg
                    if embed.ndim == 3:
                        cb = embed[0]
                    elif embed.ndim == 2:
                        cb = embed
                except AttributeError:
                    pass

            if cb is None:
                raise RuntimeError(f"Cannot extract codebook from VQ layer {q_idx}")

            codebooks.append(cb.detach().to(device_).float())

        return codebooks

    @torch.no_grad()
    def adc_maxsim_chunked(q_norm, doc_indices, doc_mask, codebooks, chunk_size=1024):
        n_docs = doc_indices.shape[0]
        NQ = doc_indices.shape[2]
        N_q = q_norm.shape[0]

        lut_list = []
        for q_idx in range(NQ):
            cb = codebooks[q_idx]
            lut_list.append(torch.mm(q_norm, cb.t()))

        all_scores = torch.zeros(n_docs, device=q_norm.device, dtype=torch.float32)

        for start in range(0, n_docs, chunk_size):
            end = min(start + chunk_size, n_docs)
            chunk_idx = doc_indices[start:end]
            chunk_mask = doc_mask[start:end]
            cs = end - start
            max_len = chunk_idx.shape[1]

            sim = torch.zeros(N_q, cs, max_len, device=q_norm.device, dtype=torch.float32)
            for q_idx in range(NQ):
                lut_q = lut_list[q_idx]
                idx_q = chunk_idx[:, :, q_idx].long()
                idx_exp = idx_q.unsqueeze(0).expand(N_q, -1, -1)
                gathered = torch.gather(
                    lut_q.unsqueeze(1).expand(-1, cs, -1),
                    dim=2,
                    index=idx_exp,
                )
                sim += gathered

            sim.masked_fill_(~chunk_mask.unsqueeze(0), float('-inf'))
            all_scores[start:end] = sim.max(dim=-1).values.sum(dim=0)

        return all_scores

    @torch.no_grad()
    def rerank_exact_maxsim(q_norm, candidate_indices, all_page_embeddings_, device_):
        K = len(candidate_indices)
        if K == 0:
            return []

        arrays = []
        for idx in candidate_indices:
            emb = all_page_embeddings_[idx]
            arrays.append(torch.from_numpy(emb.astype(np.float32)))

        max_len = max(a.shape[0] for a in arrays)
        D = arrays[0].shape[1]
        mini_mat = torch.zeros(K, max_len, D, dtype=torch.float32)
        mini_mask = torch.zeros(K, max_len, dtype=torch.bool)

        for i, a in enumerate(arrays):
            L = a.shape[0]
            mini_mat[i, :L] = F.normalize(a, dim=-1)
            mini_mask[i, :L] = True

        mini_mat = mini_mat.to(device_)
        mini_mask = mini_mask.to(device_)

        sim = torch.einsum('qd,nld->qnl', q_norm, mini_mat)
        sim.masked_fill_(~mini_mask.unsqueeze(0), float('-inf'))
        scores = sim.max(dim=-1).values.sum(dim=0)

        top_k = min(10, K)
        local_top = torch.topk(scores, top_k).indices.cpu().tolist()
        return [candidate_indices[i] for i in local_top]

    rvq_metrics = {}
    rvq_domain_metrics = {}
    rvq_query_rows = []
    rvq_latency = LatencyTracker("RVQ ADC")
    rvq_compression_info = []

    print(f"Sampling {RVQ_TRAINING_SAMPLES:,} patch vectors for codebook training...")
    train_data = _sample_patch_embeddings(all_page_embeddings, RVQ_TRAINING_SAMPLES)
    train_data = F.normalize(train_data, dim=-1)
    print(f"  Training data: {train_data.shape}")

    doc_lengths = [e.shape[0] for e in all_page_embeddings]
    max_doc_len = max(doc_lengths)
    total_patches = sum(doc_lengths)
    bytes_per_patch_f32 = EMB_DIM * 4

    for cfg_idx, (NQ, CB_SIZE, cfg_label) in enumerate(RVQ_CONFIGS):
        bytes_per_patch_rvq = NQ * (1 if CB_SIZE <= 256 else 2)
        compression_ratio = bytes_per_patch_f32 / bytes_per_patch_rvq
        mem_rvq_mb = total_patches * bytes_per_patch_rvq / 1e6
        mem_f32_mb = total_patches * bytes_per_patch_f32 / 1e6

        print(f"\n{'='*70}")
        print(f"[{cfg_idx+1}/{len(RVQ_CONFIGS)}] {cfg_label}")
        print(f"  NQ={NQ}, CB={CB_SIZE} -> {bytes_per_patch_rvq} B/patch -> {compression_ratio:.0f}x compression")
        print(f"  Index size: {mem_rvq_mb:.1f} MB (RVQ) vs {mem_f32_mb:.1f} MB (FP32)")
        print(f"{'='*70}")

        rvq_compression_info.append({
            'config': cfg_label, 'num_quantizers': NQ,
            'codebook_size': CB_SIZE, 'bytes_per_patch': bytes_per_patch_rvq,
            'compression_ratio': compression_ratio,
            'index_mb_rvq': round(mem_rvq_mb, 1),
            'index_mb_f32': round(mem_f32_mb, 1),
        })

        print("  Phase 1: Training RVQ codebooks...")
        rvq_model = ResidualVQ(
            dim=EMB_DIM,
            num_quantizers=NQ,
            codebook_size=CB_SIZE,
            kmeans_init=True,
            threshold_ema_dead_code=2,
        ).to(device)
        rvq_model.train()

        n_train = train_data.shape[0]
        for epoch in range(RVQ_TRAINING_EPOCHS):
            perm = torch.randperm(n_train)
            epoch_loss = 0.0
            n_batches = 0
            for i in range(0, n_train, RVQ_TRAINING_BATCH_SIZE):
                batch = train_data[perm[i:i+RVQ_TRAINING_BATCH_SIZE]].to(device)
                _, _, commit_loss = rvq_model(batch.unsqueeze(1))
                epoch_loss += commit_loss.sum().item()
                n_batches += 1
            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"    Epoch {epoch+1}/{RVQ_TRAINING_EPOCHS}: commit_loss={epoch_loss/max(n_batches,1):.6f}")

        rvq_model.eval()
        codebooks = _extract_codebooks(rvq_model, NQ, device)
        print(f"  Codebooks: {len(codebooks)} x {codebooks[0].shape}")

        print("  Phase 2: Quantizing index...")
        idx_dtype = np.uint8 if CB_SIZE <= 256 else np.uint16
        quantized_index = np.zeros((n_pages_total, max_doc_len, NQ), dtype=idx_dtype)
        quantized_mask = np.zeros((n_pages_total, max_doc_len), dtype=bool)

        for page_i in tqdm(range(n_pages_total), desc="  Quantizing", leave=False):
            emb = all_page_embeddings[page_i]
            L = emb.shape[0]
            quantized_mask[page_i, :L] = True

            page_t = torch.from_numpy(emb.astype(np.float32)).to(device)
            page_t = F.normalize(page_t, dim=-1)

            for s in range(0, L, RVQ_QUANTIZE_BATCH_SIZE):
                e = min(s + RVQ_QUANTIZE_BATCH_SIZE, L)
                with torch.no_grad():
                    _, indices, _ = rvq_model(page_t[s:e].unsqueeze(0))
                quantized_index[page_i, s:e, :] = indices[0].cpu().numpy().astype(idx_dtype)

        doc_indices_t = torch.from_numpy(quantized_index).to(device)
        doc_mask_t = torch.from_numpy(quantized_mask).to(device)
        del quantized_index, quantized_mask
        gc.collect()

        rerank_label = f"{cfg_label}+rerank{RERANK_TOP_K}"
        print(f"  Phase 3: ADC MaxSim + Re-ranking (top-{RERANK_TOP_K})...")

        for q_idx, item in tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc=f"  Eval {cfg_label}", leave=False):
            question = item['question']
            gt_set = item.get('gt_relevance', item['gt_embed_indices'])
            domain = item['domain']

            q_inputs = query_processor.process_queries([question]).to(device)
            with torch.no_grad():
                q_proj = query_model(**q_inputs)

            attn_mask = q_inputs['attention_mask'][0]
            trad_idx = torch.where(attn_mask > 0)[0]
            q_emb = q_proj[0][trad_idx].float()
            q_norm = F.normalize(q_emb, dim=-1)

            if M7_MEASURE_LATENCY and torch.cuda.is_available():
                torch.cuda.synchronize()
            t0 = time.perf_counter()

            scores = adc_maxsim_chunked(
                q_norm, doc_indices_t, doc_mask_t, codebooks,
                chunk_size=ADC_CHUNK_SIZE,
            )

            top10_adc = torch.topk(scores, min(10, n_pages_total)).indices.cpu().tolist()

            if M7_MEASURE_LATENCY and torch.cuda.is_available():
                torch.cuda.synchronize()
            score_ms_adc = (time.perf_counter() - t0) * 1000.0

            top_k_candidates = torch.topk(scores, min(RERANK_TOP_K, n_pages_total)).indices.cpu().tolist()

            if M7_MEASURE_LATENCY and torch.cuda.is_available():
                torch.cuda.synchronize()
            t1 = time.perf_counter()

            top10_reranked = rerank_exact_maxsim(
                q_norm, top_k_candidates, all_page_embeddings, device
            )

            if M7_MEASURE_LATENCY and torch.cuda.is_available():
                torch.cuda.synchronize()
            score_ms_rerank = (time.perf_counter() - t1) * 1000.0

            rvq_latency.add_ratio(float(NQ), score_ms_adc)

            m_adc = hit_metrics(top10_adc, gt_set)
            record(rvq_metrics, rvq_domain_metrics, cfg_label, m_adc, domain)

            m_rr = hit_metrics(top10_reranked, gt_set)
            record(rvq_metrics, rvq_domain_metrics, rerank_label, m_rr, domain)

            rvq_query_rows.append({
                'query_id': q_idx, 'doc_name': item['doc_name'],
                'domain': domain, 'question': question,
                'rvq_config': cfg_label,
                'adc_r@1': m_adc['r1'], 'adc_r@5': m_adc['r5'], 'adc_r@10': m_adc['r10'],
                'adc_ndcg@10': round(m_adc['n10'], 4),
                'rr_r@1': m_rr['r1'], 'rr_r@5': m_rr['r5'], 'rr_r@10': m_rr['r10'],
                'rr_ndcg@10': round(m_rr['n10'], 4),
                'adc_ms': round(score_ms_adc, 3),
                'rerank_ms': round(score_ms_rerank, 3),
            })

        del doc_indices_t, doc_mask_t, rvq_model, codebooks
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    del train_data
    gc.collect()

    METHOD_KEYS_RVQ = []
    for cfg in RVQ_CONFIGS:
        METHOD_KEYS_RVQ.append(cfg[2])
        METHOD_KEYS_RVQ.append(f"{cfg[2]}+rerank{RERANK_TOP_K}")

    print_summary(rvq_metrics, rvq_domain_metrics, METHOD_KEYS_RVQ,
                  title="RVQ Compression Results (ADC-only vs ADC+Rerank)")

    pd.DataFrame(rvq_query_rows).to_csv(
        os.path.join(WORKING_DIR, "rvq_queries.csv"), index=False)
    pd.DataFrame(rvq_compression_info).to_csv(
        os.path.join(WORKING_DIR, "rvq_compression_info.csv"), index=False)
    print("\n✅ Saved: rvq_queries.csv, rvq_compression_info.csv")
else:
    print(">>> METHOD 7 skipped (ENABLE_METHOD_7_RVQ=False)")

In [ ]:
# ==============================================================================
# FINAL SUMMARY — aggregate results from all methods
# Safe khi chạy bất kỳ tổ hợp method nào (1..7)
# ==============================================================================

print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

def save_summary_csv(metrics, domain_metrics, method_keys, prefix):
    rows = []
    for key in method_keys:
        if key not in metrics:
            continue
        m = metrics[key]
        cnt = m['count'] or 1
        rows.append({
            'method': key,
            'r@1': round(m['r1'] / cnt * 100, 4),
            'r@5': round(m['r5'] / cnt * 100, 4),
            'r@10': round(m['r10'] / cnt * 100, 4),
            'ndcg@1': round(m['n1'] / cnt, 6),
            'ndcg@5': round(m['n5'] / cnt, 6),
            'ndcg@10': round(m['n10'] / cnt, 6),
            'count': cnt,
        })
    df_sum = pd.DataFrame(rows)
    df_sum.to_csv(os.path.join(WORKING_DIR, f"{prefix}_summary.csv"), index=False)

    dom_rows = []
    for domain in sorted(domain_metrics):
        dm = domain_metrics[domain]
        row = {'domain': domain}
        for key in method_keys:
            m_ = dm.get(key, _init_metric())
            cnt_ = m_['count'] or 1
            row[f'{key}_ndcg10'] = round(m_['n10'] / cnt_, 6)
            row[f'{key}_r10'] = round(m_['r10'] / cnt_ * 100, 4)
        dom_rows.append(row)
    pd.DataFrame(dom_rows).to_csv(
        os.path.join(WORKING_DIR, f"{prefix}_domain.csv"), index=False
    )

    print(f"✅ Saved: {prefix}_summary.csv and {prefix}_domain.csv")

if 'trad_metrics' in dir() and trad_metrics:
    print_summary(trad_metrics, trad_domain_metrics, ['traditional'],
                  title="Traditional MaxSim")
    save_summary_csv(trad_metrics, trad_domain_metrics, ['traditional'], "traditional")

if 'hier_metrics' in dir() and hier_metrics:
    _hier_keys = METHOD_KEYS_HIER if 'METHOD_KEYS_HIER' in dir() else list(hier_metrics.keys())
    print_summary(hier_metrics, hier_domain_metrics, _hier_keys,
                  title="Hierarchical Ward Pooling")
    save_summary_csv(hier_metrics, hier_domain_metrics, _hier_keys, "hierarchical")

if 'ours_metrics' in dir() and ours_metrics:
    _ours_keys = ABLATION_KEYS_OURS if 'ABLATION_KEYS_OURS' in dir() else list(ours_metrics.keys())
    print_summary(ours_metrics, ours_domain_metrics, _ours_keys,
                  title="Ours — SVD + Cluster Pool")
    save_summary_csv(ours_metrics, ours_domain_metrics, _ours_keys, "ours_ablation")

if 'attn_metrics' in dir() and attn_metrics:
    _attn_keys = METHOD_KEYS_ATTN if 'METHOD_KEYS_ATTN' in dir() else list(attn_metrics.keys())
    print_summary(attn_metrics, attn_domain_metrics, _attn_keys,
                  title="Attention Score Pruning")
    save_summary_csv(attn_metrics, attn_domain_metrics, _attn_keys, "attention_pruning")

if 'kmeans_metrics' in dir() and kmeans_metrics:
    _km_keys = METHOD_KEYS_KMEANS if 'METHOD_KEYS_KMEANS' in dir() else list(kmeans_metrics.keys())
    print_summary(kmeans_metrics, kmeans_domain_metrics, _km_keys,
                  title="Spherical KMeans Pooling")
    save_summary_csv(kmeans_metrics, kmeans_domain_metrics, _km_keys, "spherical_kmeans")

if 'rand_metrics' in dir() and rand_metrics:
    _rnd_keys = METHOD_KEYS_RAND if 'METHOD_KEYS_RAND' in dir() else list(rand_metrics.keys())
    print_summary(rand_metrics, rand_domain_metrics, _rnd_keys,
                  title="Random Token Pruning")
    save_summary_csv(rand_metrics, rand_domain_metrics, _rnd_keys, "random_pruning")

if 'rvq_metrics' in dir() and rvq_metrics:
    _rvq_keys = METHOD_KEYS_RVQ if 'METHOD_KEYS_RVQ' in dir() else list(rvq_metrics.keys())
    print_summary(rvq_metrics, rvq_domain_metrics, _rvq_keys,
                  title="RVQ Compression (ADC and ADC+Rerank)")
    save_summary_csv(rvq_metrics, rvq_domain_metrics, _rvq_keys, "rvq_compression")

print("\n>>> All evaluations complete.")